# XMOFS-Deep: Breast cancer (TCGA-BRCA), one consistent full run on Colab
Run the cells **one by one** (Shift+Enter), top to bottom. Do NOT use "Run all".

**Before cell 1:** Runtime -> Change runtime type -> **T4 GPU** -> Save, then **Connect**.
Upload to the Files panel (left folder icon) and wait until the spinner disappears:
1. `xmofs_deep_code.zip` (the newest one)
2. `X_expression_clean.csv.gz` (108 MB, faster) **or** `X_expression_clean.csv` (300 MB)
3. `y_pam50_labels.csv`

## Cell 1: setup (prints the GPU name; you must see a GPU line and CUDA True)

In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymoo", "shap", "gseapy"], check=True)
subprocess.run("unzip -o -q xmofs_deep_code.zip", shell=True, check=True)
print(subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout or "NO GPU FOUND: set the runtime type to T4 GPU")
import torch; print("CUDA available:", torch.cuda.is_available())
print(len([f for f in os.listdir(".") if f.endswith(".py")]), "scripts ready (expect 13)")

def save(tag):
    # zip the results (without big model files) and download them to your computer
    subprocess.run(f'zip -r -q results_brca_{tag}.zip results_brca -x "*.pt"', shell=True)
    from google.colab import files
    files.download(f"results_brca_{tag}.zip")

## Cell 2: data check. It must end with DATA OK (it stops if a file is cut or wrong)

In [ ]:
import os, subprocess
import pandas as pd
if os.path.exists("X_expression_clean.csv.gz") and not os.path.exists("X_expression_clean.csv"):
    subprocess.run(["gunzip", "-k", "X_expression_clean.csv.gz"], check=True)
X = pd.read_csv("X_expression_clean.csv", index_col=0)
y = pd.read_csv("y_pam50_labels.csv", index_col=0)["label"]
print("X:", X.shape, "| y:", y.shape)
print(y.value_counts().to_string())
assert X.shape == (981, 17677), f"X has shape {X.shape}, expected (981, 17677): re-upload and wait for the spinner to finish"
assert len(y) == 981 and (X.index == y.index).all(), "X and y do not line up"
assert X.isna().sum().sum() == 0, "X contains missing values"
print("DATA OK")
del X

## Optional cell: quick smoke test (you can skip it; the pipeline already ran on OV)
Only checks that every stage runs. **Never report these numbers.**

In [ ]:
!python xmofs_runner.py --dataset brca --profile quick

## Cells 3-6: the real (full) run
Keep this tab open. Every cell downloads a zip when it finishes: keep those zips.

### Cell 3: baselines (stages 3 and 4: this gives you your OWN plain-deep-net numbers)

In [ ]:
!python xmofs_runner.py --dataset brca --profile full --stages 3,4
save("stages3-4")

### Cell 4: NSGA-II search (stage 6)

In [ ]:
!python xmofs_runner.py --dataset brca --profile full --stages 6
save("stage6")

### Cell 5: hypervolume comparison (stage 7, now also with the log-scale check)

In [ ]:
!python xmofs_runner.py --dataset brca --profile full --stages 7
save("stage7")

### Cell 6: SHAP and repeated-split statistics (stages 8 and 9)

In [ ]:
!python xmofs_runner.py --dataset brca --profile full --stages 8,9
!cat results_brca/chosen_point.json
save("stages8-9")

## Cell 7: results summary

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("results_brca/FINAL_SUMMARY.md").read()))

## Optional cell: pathway enrichment again (only if Stage 8 printed a 429 error)

In [ ]:
!sed -i 's#stage8_results/#results_brca/stage8_results/#g' stage8_enrichment_only.py
!python stage8_enrichment_only.py

## Optional cell: restore after a disconnect
Re-run cells 1 and 2, upload the last `results_brca_*.zip` you downloaded, then run this cell, then run the stage cell that was interrupted. Finished stages are skipped.

In [ ]:
import glob, subprocess, os
z = sorted(glob.glob("results_brca_*.zip"))
print("found:", z)
if z:
    subprocess.run(["unzip", "-o", "-q", z[-1]])
    print(sorted(os.listdir("results_brca")))
else:
    print("No results zip uploaded yet.")